In [3]:
!pip install genaibook

In [4]:
import torch
from diffusers import StableDiffusionXLImg2ImgPipeline
from genaibook.core import get_device
device = get_device()
# Load the pipeline
img2img_pipeline = StableDiffusionXLImg2ImgPipeline.from_pretrained(
 "stabilityai/stable-diffusion-xl-refiner-1.0",
 torch_dtype=torch.float16,
 variant="fp16",
)

Loading pipeline components...:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/517 [00:00<?, ?it/s]

In [ ]:
from genaibook.core import SampleURL, load_image, image_grid
# Load the image
url = SampleURL.ToyAstronauts
init_image = load_image(url)
prompt = "Astronaut in a jungle, cold color palette, muted colors, detailed, 8k"
# Pass the prompt and the image through the pipeline
image = img2img_pipeline(prompt, image=init_image, strength=0.5).images[0]
image_grid([init_image, image], rows=1, cols=2)

There are modules in AutoencoderKL that should be kept in float32: []. Casting directly with `to()` can lead to inconsistent results; set `torch_dtype` in `from_pretrained()` instead to keep these modules in float32.
There are modules in AutoencoderKL that should be kept in float32: []. Casting directly with `to()` can lead to inconsistent results; set `torch_dtype` in `from_pretrained()` instead to keep these modules in float32.


  0%|          | 0/25 [00:00<?, ?it/s]

In [1]:
from diffusers import StableDiffusionXLInpaintPipeline
# Load the pipeline
inpaint_pipeline = StableDiffusionXLInpaintPipeline.from_pretrained(
 "stabilityai/stable-diffusion-xl-base-1.0",
 torch_dtype=torch.float16,
 variant="fp16",
).to(device)
img_url = SampleURL.DogBenchImage
mask_url = SampleURL.DogBenchMask
init_image = load_image(img_url).convert("RGB").resize((1024, 1024))
mask_image = load_image(mask_url).convert("RGB").resize((1024, 1024))
# Pass images and prompt through the pipeline
prompt = "A majestic tiger sitting on a bench"
image = inpaint_pipeline(
 prompt=prompt,
 image=init_image,
 mask_image=mask_image,
 num_inference_steps=50,
 strength=0.80,
 width=init_image.size[0],
 heigth=init_image.size[1],
).images[0]

Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


NameError: name 'torch' is not defined

In [ ]:
image_grid([init_image, mask_image, image], rows=1, cols=3)

In [ ]:
from diffusers import DiffusionPipeline
pipeline = DiffusionPipeline.from_pretrained(
 "stabilityai/stable-diffusion-xl-base-1.0",
 torch_dtype=torch.float16,
 variant="fp16",
).to(device)


In [ ]:
from compel import Compel, ReturnedEmbeddingsType
# Use the penultimate CLIP layer as it is more expressive
embeddings_type = (
 ReturnedEmbeddingsType.PENULTIMATE_HIDDEN_STATES_NON_NORMALIZED
)
compel = Compel(
 tokenizer=[pipeline.tokenizer, pipeline.tokenizer_2],
 text_encoder=[pipeline.text_encoder, pipeline.text_encoder_2],
 returned_embeddings_type=embeddings_type,
 requires_pooled=[False, True],
)



from genaibook.core import image_grid
# Prepare the prompts
prompts = []
prompts.append("a humanoid robot eating pasta")
prompts.append(
 "a humanoid+++ robot eating pasta"
) # make its humanoid characteristics a bit more pronounced
prompts.append(
 '["a humanoid robot eating pasta", "a van gogh painting"].and(0.8, 0.2)'
) # make it van gogh!
images = []
for prompt in prompts:
 # Use the same seed across generations
 generator = torch.Generator(device=device).manual_seed(1)
 # The compel library returns both the conditioning vectors
 # and the pooled prompt embeds
 conditioning, pooled = compel(prompt)
 # We pass the conditioning and pooled prompt embeds to the pipeline
 image = pipeline(
 prompt_embeds=conditioning,
 pooled_prompt_embeds=pooled,
 num_inference_steps=30,
 generator=generator,
 ).images[0]
 images.append(image)
image_grid(images, rows=1, cols=3)

In [ ]:
from diffusers import SemanticStableDiffusionPipeline
semantic_pipeline = SemanticStableDiffusionPipeline.from_pretrained(
 "CompVis/stable-diffusion-v1-4", torch_dtype=torch.float16, variant="fp16"
).to(device)
generator = torch.Generator(device=device).manual_seed(100)
out = semantic_pipeline(
 prompt="a photo of the face of a man",
 negative_prompt="low quality, deformed",
generator=generator,
)
out.images[0]

In [ ]:
generator = torch.Generator(device=device).manual_seed(100)
out = semantic_pipeline(
 prompt="a photo of the face of a man",
 negative_prompt="low quality, deformed",
 editing_prompt="smiling, smile",
 edit_guidance_scale=4,
 edit_warmup_steps=10,
 edit_threshold=0.99,
 edit_momentum_scale=0.3,
 edit_mom_beta=0.6,
 reverse_editing_direction=False,
 generator=generator,
)
out.images[0]

In [ ]:
generator = torch.Generator(device=device).manual_seed(100)
out = semantic_pipeline(
 prompt="a photo of the face of a man",
 negative_prompt="low quality, deformed",
 editing_prompt="glasses, wearing glasses",
 reverse_editing_direction=False,
 edit_warmup_steps=10,
 edit_guidance_scale=4,
 edit_threshold=0.99,
 edit_momentum_scale=0.3,
 edit_mom_beta=0.6,
 generator=generator,
)
out.images[0]

In [ ]:
generator = torch.Generator(device=device).manual_seed(100)
out = semantic_pipeline(
 prompt="a photo of the face of a man",
 negative_prompt="low quality, deformed",
 editing_prompt=[
 "smiling, smile",
 "glasses, wearing glasses",
 ],
 reverse_editing_direction=[False, False],
 edit_warmup_steps=[10, 10],
 edit_guidance_scale=[6, 6],
 edit_threshold=[0.99, 0.99],
 edit_momentum_scale=0.3,
 edit_mom_beta=0.6,
 generator=generator,
)
out.images[0]

In [ ]:
from diffusers import LEditsPPPipelineStableDiffusion
# Load the model as usual
pipe = LEditsPPPipelineStableDiffusion.from_pretrained(
 "stable-diffusion-v1-5/stable-diffusion-v1-5",
 torch_dtype=torch.float16,
 variant="fp16"
)
pipe.to(device)
image = load_image(SampleURL.ManInGlasses).convert("RGB")
# Invert the image, gradually adding noise to it so
# it can be denoised with modified directions,
# effectively providing an edit
pipe.invert(image=image, num_inversion_steps=50, skip=0.2)
# Edit the image with an editing prompt
edited_image = pipe(
 editing_prompt=["glasses"],
 # tell the model to remove the glasses by editing the direction
 reverse_editing_direction=[True],
 edit_guidance_scale=[1.5],
 edit_threshold=[0.95],
).images[0]
image_grid([image, edited_image], rows=1, cols=2)

In [ ]:
from diffusers import (
 EDMEulerScheduler,
 StableDiffusionXLInstructPix2PixPipeline,
)
from huggingface_hub import hf_hub_download
edit_file = hf_hub_download(
 repo_id="stabilityai/cosxl", filename="cosxl_edit.safetensors"
)
# from_single_file loads a diffusion model from a single diffusers file
pipe_edit = StableDiffusionXLInstructPix2PixPipeline.from_single_file(
 edit_file, num_in_channels=8, is_cosxl_edit=True, torch_dtype=torch.float16
)
# The model was trained so that the EDMEulerScheduler
# is the correct noise scheduler for denoising
pipe_edit.scheduler = EDMEulerScheduler(
 sigma_min=0.002,
 sigma_max=120.0,
 sigma_data=1.0,
 prediction_type="v_prediction",
 sigma_schedule="exponential",
)
pipe_edit.to(device)
prompt = "make it a cloudy day"
image = load_image(SampleURL.Mountain)
edited_image = pipe_edit(
 prompt=prompt, image=image, num_inference_steps=20
).images[0]
image_grid([image, edited_image], rows=1, cols=2)

In [ ]:
from diffusers import ControlNetModel, StableDiffusionXLControlNetPipeline
controlnet = ControlNetModel.from_pretrained(
 "diffusers/controlnet-depth-sdxl-1.0",
 torch_dtype=torch.float16,
 variant="fp16",
)
controlnet_pipeline = StableDiffusionXLControlNetPipeline.from_pretrained(
 "stabilityai/stable-diffusion-xl-base-1.0",
 controlnet=controlnet,
 torch_dtype=torch.float16,
 variant="fp16",
)
controlnet_pipeline.enable_model_cpu_offload() # Optional, saves VRAM
controlnet_pipeline.to(device)

In [ ]:
from controlnet_aux import MidasDetector
from PIL import Image
original_image = load_image(SampleURL.WomanSpeaking)
original_image = original_image.resize((1024, 1024))
# loads the MiDAS depth detector model
midas = MidasDetector.from_pretrained("lllyasviel/Annotators")
# Apply MiDAS depth detection
processed_image_midas = midas(original_image).resize(
 (1024, 1024), Image.BICUBIC
)


In [ ]:
image = controlnet_pipeline(
 "A colorful, ultra-realistic masked super hero singing a song",
 image=processed_image_midas,
 controlnet_conditioning_scale=0.4,
 num_inference_steps=30,
).images[0]
image_grid([original_image, processed_image_midas, image], rows=1, cols=3)

In [ ]:
from diffusers import StableDiffusionXLPipeline
sdxl_base_pipeline = StableDiffusionXLPipeline.from_pretrained(
 "stabilityai/stable-diffusion-xl-base-1.0",
 torch_dtype=torch.float16,
 variant="fp16",
)
sdxl_base_pipeline.to(device)
# We load the IP-Adapter too
sdxl_base_pipeline.load_ip_adapter(
 "h94/IP-Adapter", subfolder="sdxl_models", weight_name="ip-adapter_sdxl.bin"
)
# We can set the scale of how strong we
# want our IP-Adapter to impact our overall result
sdxl_base_pipeline.set_ip_adapter_scale(0.8)
image = load_image(SampleURL.ItemsVariation)
original_image = image.resize((1024, 1024))
# Create the image variation
generator = torch.Generator(device=device).manual_seed(1)
variation_image = sdxl_base_pipeline(
 prompt="",
 ip_adapter_image=original_image,
 num_inference_steps=25,
 generator=generator,
).images
image_grid([original_image, variation_image[0]], rows=1, cols=2)

In [ ]:
# We load the model and the IP-Adapter, just as before
pipeline = StableDiffusionXLPipeline.from_pretrained(
 "stabilityai/stable-diffusion-xl-base-1.0", torch_dtype=torch.float16
).to(device)
# Load the IP-Adapter into the model
pipeline.load_ip_adapter(
 "h94/IP-Adapter", subfolder="sdxl_models", weight_name="ip-adapter_sdxl.bin"
)
# We are applying the IP-Adapter only to the mid block,
# which is where it should be mapped to the style in SDXL
scale = {"up": {"block_0": [0.0, 1.0, 0.0]}}
pipeline.set_ip_adapter_scale(scale)
image = load_image(SampleURL.Mamoeiro)
original_image = image.resize((1024, 1024))
# Run inference to generate the stylized image
generator = torch.Generator(device=device).manual_seed(0)
variation_image = pipeline(
 prompt="a cat inside of a box",
 ip_adapter_image=original_image,
 num_inference_steps=25,
 generator=generator,
).images
image_grid([original_image, variation_image[0]], rows=1, cols=2)

In [ ]:
controlnet = ControlNetModel.from_pretrained(
 "diffusers/controlnet-depth-sdxl-1.0", torch_dtype=torch.float16
)
# Load the ControlNet pipeline
controlnet_pipeline = StableDiffusionXLControlNetPipeline.from_pretrained(
 "stabilityai/stable-diffusion-xl-base-1.0",
 controlnet=controlnet,
 torch_dtype=torch.float16,
 variant="fp16",
)
controlnet_pipeline.to(device)
# Load the IP-Adapter
controlnet_pipeline.load_ip_adapter(
 "h94/IP-Adapter", subfolder="sdxl_models", weight_name="ip-adapter_sdxl.bin"
)
# We are applying the IP-Adapter only to the mid block,
# which is where it should be mapped to the style in SDXL
scale = {
 "up": {"block_0": [0.0, 1.0, 0.0]},
}
controlnet_pipeline.set_ip_adapter_scale(scale)
# Load the original image
original_image = load_image(SampleURL.WomanSpeaking)
original_image = original_image.resize((1024, 1024))
# Load the style image
style_image = load_image(SampleURL.Mamoeiro)
style_image = style_image.resize((1024, 1024))
# Apply the MiDAS depth estimation
processed_image_midas = midas(original_image).resize(
 (1024, 1024), Image.BICUBIC
)
image = controlnet_pipeline(
 "A masked super hero singing a song",
 image=processed_image_midas,
 ip_adapter_image=style_image,
 controlnet_conditioning_scale=0.5,
).images[0]
image_grid(
 [original_image, style_image, processed_image_midas, image], rows=1, cols=4
)